# SondeHub download on Colab

Fetches real balloon flight tracks from eight launch sites ringing Austin (the four weather-service stations we
already use, plus Lake Charles, Shreveport, a College Station site and a Beaumont-area site), reduces each to a
small file, and saves them to Drive as zipped batches. Nothing is kept raw: a raw track can be hundreds of MB.

Use a **CPU** runtime (not a GPU). Expect roughly 15,000 flights and a few hours; the run is safe to stop and
rerun (finished flights are listed in a manifest and skipped).

Before running: regenerate and upload `balloon_code.zip` (`python -m scripts.make_colab_bundle`, then copy
`dist/balloon_code.zip` into the `balloon-forecaster` Drive folder, replacing the old one). It also carries the
list of tracks the laptop already has, so those are not fetched again.

Run the cells in order.

## 1. Mount Drive and unpack the code

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/balloon-forecaster'
CODE_DIR = '/content/code'
OUT_DIR = f'{DATA_ROOT}/sondehub_batches'
assert os.path.isdir(DATA_ROOT), f'{DATA_ROOT} not found: add the shared folder shortcut to My Drive'
bundle = f'{DATA_ROOT}/balloon_code.zip'
assert os.path.exists(bundle), 'Upload balloon_code.zip into the balloon-forecaster Drive folder first'
zipfile.ZipFile(bundle).extractall(CODE_DIR)
assert os.path.exists(f'{CODE_DIR}/scripts/fetch_sondehub_compact.py'), 'old balloon_code.zip: rebuild and upload it'
print('code unpacked; CPUs:', os.cpu_count())

In [ ]:
!pip -q install orjson

## 2. Small test first (College Station site, 17 flights, about a minute)
Check that it prints `ok` counts and writes a zip. The test uses its own scratch folder, not Drive.

In [ ]:
!cd {CODE_DIR} && python -u -m scripts.fetch_sondehub_compact --out /content/test_out --sites CLL --workers 2 --batch 10
!ls -la /content/test_out

## 3. The full run
Eight sites, 2021-03-22 onward. `--max-gz-mb 40` skips the rare enormous tracks (they could exhaust the 12 GB of RAM).
Progress prints every batch of 400 flights. If it stops, run this cell again.

In [ ]:
import subprocess, sys, time

skip = f'{CODE_DIR}/configs/have_sondehub_serials.txt'
log = open('/content/sondehub.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-u', '-m', 'scripts.fetch_sondehub_compact', '--out', OUT_DIR, '--workers', str(os.cpu_count() or 2),
     '--max-gz-mb', '40', '--skip-list', skip],
    cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
try:
    while proc.poll() is None:
        time.sleep(120)
        lines = open('/content/sondehub.log').read().splitlines()
        print(f'{(time.time() - t0) / 60:6.0f} min | {lines[-1] if lines else ""}', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('Stopped. Rerun this cell to resume.')
print('exit code', proc.returncode)
print(''.join(open('/content/sondehub.log').readlines()[-6:]))

## 4. Flush to Drive and verify
Colab writes to Drive through a cache, so flush and remount before trusting any count.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

In [ ]:
import glob, zipfile
import pandas as pd

zips = sorted(glob.glob(f'{OUT_DIR}/tracks_*.zip'))
n_tracks, bad = 0, []
for z in zips:
    try:
        with zipfile.ZipFile(z) as zf:
            assert zf.testzip() is None
            n_tracks += len(zf.namelist())
    except Exception as e:
        bad.append((os.path.basename(z), repr(e)))
m = pd.read_csv(f'{OUT_DIR}/sondehub_manifest.csv', dtype=str)
print(len(zips), 'zips,', n_tracks, 'tracks inside,', round(sum(os.path.getsize(z) for z in zips) / 1e6), 'MB,', len(bad), 'unreadable')
print('manifest statuses:', m.status.str.split(':').str[0].value_counts().to_dict())
print('ok tracks per station:
', m[m.status == 'ok'].groupby('station').size().to_string())
print('manifest ok count matches zips:', (m.status == 'ok').sum() == n_tracks)
for name, err in bad: print('  BAD', name, err)

## 5. Bring it to the laptop
Download the Drive folder `balloon-forecaster/sondehub_batches` (it should be about 1 to 2 GB), unzip the Drive download
if it is split, and tell Claude where it is. Claude runs `scripts.import_sondehub_batches` and the validation. After the
laptop copy is verified, delete the folder from Drive and empty the Drive trash (trashed files still count).